# **YouTube Data Ingestion Pipeline**
This notebook accomplishes the following:
1. Pull official YouTube metadata via GC's API. 
2. Pull transcripts via an external API, provided a video ID. 

## Data Storage
- Right now, just storing it in a local data subdirectory
- Need to migrate ingested data to Supabase

In [67]:
import os
from dotenv import load_dotenv
from googleapiclient.discovery import build

# Load .env file
load_dotenv()
YT_API_KEY = os.getenv("YOUTUBE_DATA_API_KEY")
# print(YT_API_KEY)

## Query via YouTube Service Object
Run a simple search query from this.

In [68]:
youtube = build(serviceName="youtube", version="v3", developerKey=YT_API_KEY)

To Do: 
- Implement some more specific search queries on video search (views, impressions, etc.)

In [69]:
# Returns a nested dictionary
response = youtube.search().list(
    q="personal health",
    part="snippet",
    type="video",
    maxResults=5
).execute()

In [70]:
print("Search query fields:\n--------------------")
for k in response.keys():
    print(f"{k}")

Search query fields:
--------------------
kind
etag
nextPageToken
regionCode
pageInfo
items


In [71]:
print(type(response["items"]))
print(response["items"])


<class 'list'>
[{'kind': 'youtube#searchResult', 'etag': 'fr1S--ADQhg5XRx1VD_1Z8ad5FM', 'id': {'kind': 'youtube#video', 'videoId': '2-8b2uZBfe8'}, 'snippet': {'publishedAt': '2026-01-24T16:00:06Z', 'channelId': 'UCHQ4lSaKRap5HyrpitrTOhQ', 'title': 'A Personal Health Share That Might Help You', 'description': 'Embrace Breakthroughs With The “Heal From a Breakup” Course. Free for Life With a 7-Day Trial.', 'thumbnails': {'default': {'url': 'https://i.ytimg.com/vi/2-8b2uZBfe8/default.jpg', 'width': 120, 'height': 90}, 'medium': {'url': 'https://i.ytimg.com/vi/2-8b2uZBfe8/mqdefault.jpg', 'width': 320, 'height': 180}, 'high': {'url': 'https://i.ytimg.com/vi/2-8b2uZBfe8/hqdefault.jpg', 'width': 480, 'height': 360}}, 'channelTitle': 'Thais Gibson - Personal Development School', 'liveBroadcastContent': 'none', 'publishTime': '2026-01-24T16:00:06Z'}}, {'kind': 'youtube#searchResult', 'etag': 'Yo4O6rldRxaI23fi1dGU7ZJh3fM', 'id': {'kind': 'youtube#video', 'videoId': 'wp_k5wVn1Gs'}, 'snippet': {'p

In [72]:
# Items is a list of dictionaries
print(type(response["items"][0]))

first_item = response["items"][0]

<class 'dict'>


In [73]:
print(f"Keys under Each Item:\n{first_item.keys()}")

Keys under Each Item:
dict_keys(['kind', 'etag', 'id', 'snippet'])


In [74]:
print("Key | Value")
print("------------")
for k in first_item.keys():
    print(f"{k} ----- {first_item[k]}")

Key | Value
------------
kind ----- youtube#searchResult
etag ----- fr1S--ADQhg5XRx1VD_1Z8ad5FM
id ----- {'kind': 'youtube#video', 'videoId': '2-8b2uZBfe8'}
snippet ----- {'publishedAt': '2026-01-24T16:00:06Z', 'channelId': 'UCHQ4lSaKRap5HyrpitrTOhQ', 'title': 'A Personal Health Share That Might Help You', 'description': 'Embrace Breakthroughs With The “Heal From a Breakup” Course. Free for Life With a 7-Day Trial.', 'thumbnails': {'default': {'url': 'https://i.ytimg.com/vi/2-8b2uZBfe8/default.jpg', 'width': 120, 'height': 90}, 'medium': {'url': 'https://i.ytimg.com/vi/2-8b2uZBfe8/mqdefault.jpg', 'width': 320, 'height': 180}, 'high': {'url': 'https://i.ytimg.com/vi/2-8b2uZBfe8/hqdefault.jpg', 'width': 480, 'height': 360}}, 'channelTitle': 'Thais Gibson - Personal Development School', 'liveBroadcastContent': 'none', 'publishTime': '2026-01-24T16:00:06Z'}


In [75]:
for item in response["items"]:
    print(item["snippet"]["title"])
    print(item["snippet"]["channelTitle"])
    print("------------")

A Personal Health Share That Might Help You
Thais Gibson - Personal Development School
------------
My 5 Unconventional Healthy Habits
Keltie O'Connor
------------
Hygiene Habits for Kids - Compilation - Handwashing, Personal Hygiene and Tooth Brushing
Smile and Learn - English
------------
A personal health coach for those living with chronic diseases | Priscilla Pemu
TED
------------
Wellbeing for Children: Healthy Habits
ClickView
------------


To Do: 
- Get each video ID from the Data API's response:
    - pass as arg into transcript extraction API
    - save as a .txt file in data/transcripts/

In [76]:
list_of_video_ids = []
for item in response["items"]:
    list_of_video_ids.append(item['id']['videoId'])

In [77]:
print("List of Video IDs\n-----------------")
for i in list_of_video_ids:
    print(i)

List of Video IDs
-----------------
2-8b2uZBfe8
wp_k5wVn1Gs
l6XGE-Xuq3M
i5vZkaUk528
dhpCdqOtuj0


## Transcript Extraction
Provided video IDs, extract transcript

In [78]:
# Initalize Transcript Api
from youtube_transcript_api import YouTubeTranscriptApi

ytt_api = YouTubeTranscriptApi()

# Outputs the RAW text source structure; must store this for source tracking in DB;
raw_transcript = ytt_api.fetch(list_of_video_ids[0])

# Returns a nested object
print(raw_transcript)

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='I felt like I was just always sick in my', start=0.0, duration=3.04), FetchedTranscriptSnippet(text='childhood. I was always covered in like', start=1.6, duration=3.36), FetchedTranscriptSnippet(text='rashes and hives and I was allergic to', start=3.04, duration=4.0), FetchedTranscriptSnippet(text='everything. Got addicted to painkillers', start=4.96, duration=3.599), FetchedTranscriptSnippet(text='right after knee surgery. Went through', start=7.04, duration=2.96), FetchedTranscriptSnippet(text='like a six and a half year battle with', start=8.559, duration=3.841), FetchedTranscriptSnippet(text='that. Started like having these rashes', start=10.0, duration=4.16), FetchedTranscriptSnippet(text='come back on my arms like these kind of', start=12.4, duration=4.0), FetchedTranscriptSnippet(text='chronic hives. I would wake up in the', start=14.16, duration=4.32), FetchedTranscriptSnippet(text='morning with like profound brain fog.

In [79]:
import os
from youtube_transcript_api.formatters import JSONFormatter

# Ensure the directory exists
os.makedirs('data/transcripts', exist_ok=True)

# Format the raw transcript as JSON using the built-in formatter
formatter = JSONFormatter()
json_formatted = formatter.format_transcript(raw_transcript, indent=2)

# Save the raw transcript as JSON to a file
with open('data/transcripts/raw.json', 'w', encoding='utf-8') as f:
    f.write(json_formatted)

print("Raw transcript saved to data/transcripts/raw.json")

Raw transcript saved to data/transcripts/raw.json


## Transcript Text Parsing

In [80]:
import re
from typing import Union, List, Dict

def clean_transcript(transcript_data: Union[List[Dict], str]) -> str:
    """
    Clean a YouTube transcript by removing noise and formatting artifacts.
    
    Args:
        transcript_data: Either a list of transcript dicts from YouTubeTranscriptApi
                        (with 'text' keys) or a raw string
    
    Returns:
        A clean, readable transcript as a single string
    """
    # Handle both FetchedTranscript objects and raw strings
    if isinstance(transcript_data, str):
        text = transcript_data
    else:
        # Join text from transcript items (FetchedTranscriptSnippet objects have .text attribute)
        text = " ".join([item.text for item in transcript_data])
    
    # Remove speaker indicators (>>, Speaker: patterns)
    text = re.sub(r'^>\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'(?:^|\s)(?:[A-Z][a-z]*(?:\s+[A-Z][a-z]*)?)\s*:\s*', ' ', text)
    
    # Remove sound/action tags like [music], [snorts], [applause], etc.
    text = re.sub(r'\[[^\]]*\]', '', text, flags=re.IGNORECASE)
    
    # Remove parenthetical tags like (audience laughs)
    text = re.sub(r'\([^)]*\)', '', text)
    
    # Remove filler words strategically (avoid removing semantically important ones)
    filler_patterns = [
        r'\b(?:um|uh|ugh|hmm)\b',
        r'\byou\s+know\b',
        r'\blike\b(?=\s+(?:he|she|they|it|the|a|i))',
    ]
    for pattern in filler_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    
    # Fix excessive whitespace
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)
    
    # Fix spacing after sentence-ending punctuation
    text = re.sub(r'([.!?])\s+(?=[a-z])', lambda m: m.group(1) + ' ', text)
    
    # Clean up edge cases
    text = text.strip()
    
    # Capitalize first letter
    if text and text[0].islower():
        text = text[0].upper() + text[1:]
    
    return text

In [81]:
# Test with the raw transcript
cleaned_video_transcript = clean_transcript(raw_transcript)

print("Cleaned Transcript:\n" + "="*60)
print(cleaned_video_transcript[:500])  # Print first 500 chars
print("\n[...]")

Cleaned Transcript:
I felt I was just always sick in my childhood. I was always covered in like rashes and hives and I was allergic to everything. Got addicted to painkillers right after knee surgery. Went through a six and a half year battle with that. Started like having these rashes come back on my arms these kind of chronic hives. I would wake up in the morning with like profound brain fog. And it didn't feel it was a burnout thing cuz I loved what I was doing. I felt I still had like tons of emotional and ment

[...]


In [82]:
import os

# Ensure the directory exists
os.makedirs('data/transcripts', exist_ok=True)

# Save the cleaned transcript to a .txt file
with open('data/transcripts/cleaned.txt', 'w') as f:
    f.write(cleaned_video_transcript)